# Flight Delay Prediction
**DSC 148 Course Project**

Predicting whether a U.S. domestic flight will arrive **more than 15 minutes late** (the DOT standard for a delayed flight), using only information known *before* the flight departs.

Dataset: 2015 Flight Delays and Cancellations (U.S. DOT / Bureau of Transportation Statistics), ~5.8M flights.

**Pipeline:** EDA → target creation & leakage removal → feature engineering → baselines (Logistic Regression, Random Forest) → final model (LightGBM) → ablation, feature importance, hyperparameter sensitivity.

## 1. Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)
import lightgbm as lgb

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
RANDOM_STATE = 42

# Folders for saving outputs (these are what you'll send me for the report)
for folder in ['figures', 'results']:
    os.makedirs(folder, exist_ok=True)

## 2. Load the Data

Make sure the three CSVs are inside a `data/` folder next to this notebook.

In [ ]:
flights = pd.read_csv('data/flights.csv', low_memory=False)
airlines = pd.read_csv('data/airlines.csv')
airports = pd.read_csv('data/airports.csv')

print('Flights shape:', flights.shape)
print('Airlines shape:', airlines.shape)
print('Airports shape:', airports.shape)
flights.head()

In [ ]:
# Attach the full airline name (airlines.csv maps IATA_CODE -> full name)
airlines = airlines.rename(columns={'AIRLINE': 'AIRLINE_NAME'})
flights = flights.merge(airlines, left_on='AIRLINE', right_on='IATA_CODE', how='left')
flights[['AIRLINE', 'AIRLINE_NAME']].drop_duplicates().head()

## 3. Exploratory Data Analysis

In [ ]:
flights.info()

In [ ]:
# Missing values per column
missing = flights.isnull().sum().sort_values(ascending=False)
missing[missing > 0]

### Target creation & filtering

We predict **arrival** delay, so cancelled and diverted flights (which have no arrival) are removed. The target is `is_delayed = 1` when `ARRIVAL_DELAY > 15` minutes.

In [ ]:
df = flights[(flights['CANCELLED'] == 0) & (flights['DIVERTED'] == 0)].copy()
df = df[df['ARRIVAL_DELAY'].notnull()]
print('After removing cancelled/diverted:', df.shape)

df['is_delayed'] = (df['ARRIVAL_DELAY'] > 15).astype(int)
print(df['is_delayed'].value_counts(normalize=True).round(3))

In [ ]:
# Class distribution
plt.figure()
df['is_delayed'].value_counts().sort_index().plot(kind='bar', color=['#4C9F70', '#E07A5F'])
plt.title('Class Distribution: On-Time vs Delayed')
plt.xticks([0, 1], ['On-Time (<=15 min)', 'Delayed (>15 min)'], rotation=0)
plt.ylabel('Number of Flights')
plt.tight_layout()
plt.savefig('figures/class_distribution.png', dpi=150)
plt.show()

In [ ]:
# Delay rate by airline
airline_delay = df.groupby('AIRLINE_NAME')['is_delayed'].mean().sort_values(ascending=False)
plt.figure(figsize=(10, 7))
airline_delay.plot(kind='barh', color='#3D5A80')
plt.title('Delay Rate by Airline')
plt.xlabel('Proportion of Flights Delayed (>15 min)')
plt.tight_layout()
plt.savefig('figures/delay_by_airline.png', dpi=150)
plt.show()

In [ ]:
# Delay rate by month (seasonality)
month_delay = df.groupby('MONTH')['is_delayed'].mean()
plt.figure()
month_delay.plot(kind='line', marker='o', color='#E07A5F')
plt.title('Delay Rate by Month')
plt.xlabel('Month'); plt.ylabel('Delay Rate')
plt.xticks(range(1, 13))
plt.tight_layout()
plt.savefig('figures/delay_by_month.png', dpi=150)
plt.show()

In [ ]:
# Delay rate by scheduled departure hour
df['hour_of_day'] = df['SCHEDULED_DEPARTURE'] // 100
hour_delay = df.groupby('hour_of_day')['is_delayed'].mean()
plt.figure()
hour_delay.plot(kind='bar', color='#98C1D9')
plt.title('Delay Rate by Scheduled Departure Hour')
plt.xlabel('Hour of Day'); plt.ylabel('Delay Rate')
plt.tight_layout()
plt.savefig('figures/delay_by_hour.png', dpi=150)
plt.show()

In [ ]:
# Delay rate by day of week
dow_delay = df.groupby('DAY_OF_WEEK')['is_delayed'].mean()
plt.figure()
dow_delay.plot(kind='bar', color='#EE6C4D')
plt.title('Delay Rate by Day of Week')
plt.xlabel('Day of Week (1 = Monday ... 7 = Sunday)'); plt.ylabel('Delay Rate')
plt.tight_layout()
plt.savefig('figures/delay_by_dow.png', dpi=150)
plt.show()

In [ ]:
# Top 15 origin airports by delay rate (only busy ones, >10k flights)
airport_counts = df['ORIGIN_AIRPORT'].value_counts()
busy = airport_counts[airport_counts > 10000].index
airport_delay = (df[df['ORIGIN_AIRPORT'].isin(busy)]
                 .groupby('ORIGIN_AIRPORT')['is_delayed'].mean()
                 .sort_values(ascending=False).head(15))
plt.figure(figsize=(10, 7))
airport_delay.plot(kind='barh', color='#293241')
plt.gca().invert_yaxis()
plt.title('Top 15 Origin Airports by Delay Rate (min. 10k flights)')
plt.xlabel('Delay Rate')
plt.tight_layout()
plt.savefig('figures/delay_by_airport.png', dpi=150)
plt.show()

> **Data note:** From October–December 2015 the BTS recorded some airports as 5-digit numeric codes instead of IATA letter codes. This is a known quirk of this dataset. It doesn't affect our modeling because we encode airports by their historical delay rate (below), which works regardless of the code format.

## 4. Sampling for Tractability

The full dataset has ~5.7M usable rows. Logistic Regression and Random Forest are slow at that scale, so we work with a random sample (still far above the 50k requirement). LightGBM could handle the full set, but we keep one sample for a fair, apples-to-apples comparison across all models.

In [ ]:
SAMPLE_SIZE = 600000   # lower this (e.g. 200000) if your machine is slow
if len(df) > SAMPLE_SIZE:
    df_model = df.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)
else:
    df_model = df.copy()
print('Modeling sample shape:', df_model.shape)

## 5. Feature Engineering

**Leakage warning:** columns like `DEPARTURE_DELAY`, `WHEELS_OFF`, `AIRLINE_DELAY`, `WEATHER_DELAY`, and the actual times are only known *after* a flight is underway. We never use them. The model only sees information available at booking/scheduling time.

In [ ]:
def time_of_day(hour):
    if 5 <= hour <= 11:  return 'Morning'
    if 12 <= hour <= 17: return 'Afternoon'
    if 18 <= hour <= 21: return 'Evening'
    return 'Night'

def season(month):
    if month in (12, 1, 2): return 'Winter'
    if month in (3, 4, 5):  return 'Spring'
    if month in (6, 7, 8):  return 'Summer'
    return 'Fall'

df_model['hour_of_day'] = df_model['SCHEDULED_DEPARTURE'] // 100
df_model['time_of_day'] = df_model['hour_of_day'].apply(time_of_day)
df_model['season']      = df_model['MONTH'].apply(season)
df_model['is_weekend']  = (df_model['DAY_OF_WEEK'] >= 6).astype(int)
df_model['SCHEDULED_TIME'] = df_model['SCHEDULED_TIME'].fillna(df_model['SCHEDULED_TIME'].median())
df_model[['hour_of_day', 'time_of_day', 'season', 'is_weekend']].head()

### Train/test split first, then target-encode airports

We split **before** computing airport delay-rate features so those rates are learned from training data only. This prevents test-set leakage.

In [ ]:
target = 'is_delayed'
y = df_model[target]

train_idx, test_idx = train_test_split(
    df_model.index, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
train = df_model.loc[train_idx].copy()
test  = df_model.loc[test_idx].copy()

# Historical delay rate per airport, computed on TRAIN ONLY
global_rate = train[target].mean()
origin_map = train.groupby('ORIGIN_AIRPORT')[target].mean()
dest_map   = train.groupby('DESTINATION_AIRPORT')[target].mean()

for d in (train, test):
    d['origin_delay_rate'] = d['ORIGIN_AIRPORT'].map(origin_map).fillna(global_rate)
    d['dest_delay_rate']   = d['DESTINATION_AIRPORT'].map(dest_map).fillna(global_rate)

print('Train:', train.shape, '| Test:', test.shape)

### Build two feature matrices

- **Base** = raw scheduling info only (for the ablation study)
- **Engineered** = base + time-of-day, season, weekend flag, and airport delay-rate features

In [ ]:
base_numeric = ['MONTH', 'DAY_OF_WEEK', 'hour_of_day', 'DISTANCE', 'SCHEDULED_TIME']
base_cat     = ['AIRLINE']

eng_numeric  = base_numeric + ['is_weekend', 'origin_delay_rate', 'dest_delay_rate']
eng_cat      = ['AIRLINE', 'time_of_day', 'season']

def build_matrix(train, test, numeric, categorical):
    Xtr = pd.get_dummies(train[numeric + categorical], columns=categorical)
    Xte = pd.get_dummies(test[numeric + categorical], columns=categorical)
    Xtr, Xte = Xtr.align(Xte, join='left', axis=1, fill_value=0)  # match columns
    return Xtr, Xte

X_train_base, X_test_base = build_matrix(train, test, base_numeric, base_cat)
X_train_eng,  X_test_eng  = build_matrix(train, test, eng_numeric, eng_cat)

y_train = train[target]
y_test  = test[target]

print('Base feature count:      ', X_train_base.shape[1])
print('Engineered feature count:', X_train_eng.shape[1])

## 6. Evaluation Helper

In [ ]:
def evaluate(name, y_true, y_pred):
    return {
        'Model': name,
        'Accuracy':  round(accuracy_score(y_true, y_pred), 4),
        'Precision': round(precision_score(y_true, y_pred), 4),
        'Recall':    round(recall_score(y_true, y_pred), 4),
        'F1':        round(f1_score(y_true, y_pred), 4),
    }

results = []

## 7. Baseline 1 — Logistic Regression

In [ ]:
scaler = StandardScaler()
X_train_eng_scaled = scaler.fit_transform(X_train_eng)
X_test_eng_scaled  = scaler.transform(X_test_eng)

lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_train_eng_scaled, y_train)
lr_pred = lr.predict(X_test_eng_scaled)

res = evaluate('Logistic Regression', y_test, lr_pred)
results.append(res)
print(res)
print(classification_report(y_test, lr_pred))

## 8. Baseline 2 — Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=20,
                            n_jobs=-1, random_state=RANDOM_STATE)
rf.fit(X_train_eng, y_train)
rf_pred = rf.predict(X_test_eng)

res = evaluate('Random Forest', y_test, rf_pred)
results.append(res)
print(res)
print(classification_report(y_test, rf_pred))

## 9. Final Model — LightGBM

In [ ]:
lgb_clf = lgb.LGBMClassifier(
    n_estimators=400,
    num_leaves=64,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
)
lgb_clf.fit(X_train_eng, y_train)
lgb_pred = lgb_clf.predict(X_test_eng)

res = evaluate('LightGBM', y_test, lgb_pred)
results.append(res)
print(res)
print(classification_report(y_test, lgb_pred))

## 10. Model Comparison

In [ ]:
results_df = pd.DataFrame(results).set_index('Model')
results_df.to_csv('results/model_comparison.csv')
results_df

In [ ]:
plt.figure()
results_df['F1'].plot(kind='bar', color='#3D5A80')
plt.title('F1 Score by Model')
plt.ylabel('F1 Score'); plt.xticks(rotation=15)
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig('figures/model_comparison.png', dpi=150)
plt.show()

## 11. Ablation Study

Same LightGBM model, trained on **base features only** vs. **base + engineered features**. The gap shows how much the engineered features actually contributed.

In [ ]:
lgb_base = lgb.LGBMClassifier(
    n_estimators=400, num_leaves=64, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE)
lgb_base.fit(X_train_base, y_train)
base_pred = lgb_base.predict(X_test_base)

ablation = pd.DataFrame([
    evaluate('LightGBM (base features)', y_test, base_pred),
    evaluate('LightGBM (+ engineered features)', y_test, lgb_pred),
]).set_index('Model')
ablation.to_csv('results/ablation.csv')
ablation

## 12. Feature Importance

In [ ]:
importances = pd.Series(lgb_clf.feature_importances_, index=X_train_eng.columns)
top = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 7))
top.plot(kind='barh', color='#293241')
plt.gca().invert_yaxis()
plt.title('LightGBM Feature Importance (Top 15)')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('figures/feature_importance.png', dpi=150)
plt.show()
top

## 13. Hyperparameter Sensitivity (num_leaves)

In [ ]:
sens = []
for nl in [16, 32, 64, 128]:
    m = lgb.LGBMClassifier(
        n_estimators=400, num_leaves=nl, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE)
    m.fit(X_train_eng, y_train)
    p = m.predict(X_test_eng)
    sens.append({'num_leaves': nl,
                 'F1': round(f1_score(y_test, p), 4),
                 'Accuracy': round(accuracy_score(y_test, p), 4)})

sens_df = pd.DataFrame(sens)
sens_df.to_csv('results/hyperparam_sensitivity.csv', index=False)

plt.figure()
plt.plot(sens_df['num_leaves'], sens_df['F1'], marker='o')
plt.title('F1 Score vs num_leaves')
plt.xlabel('num_leaves'); plt.ylabel('F1 Score')
plt.tight_layout()
plt.savefig('figures/hyperparam_sensitivity.png', dpi=150)
plt.show()
sens_df

## 14. Confusion Matrix (Final Model)

In [ ]:
cm = confusion_matrix(y_test, lgb_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['On-Time', 'Delayed'],
            yticklabels=['On-Time', 'Delayed'])
plt.title('LightGBM Confusion Matrix')
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('figures/confusion_matrix.png', dpi=150)
plt.show()

## 15. Done

Everything you need for the report has been saved:
- **`figures/`** — all plots (class distribution, EDA charts, model comparison, feature importance, confusion matrix, hyperparameter curve)
- **`results/`** — CSVs: `model_comparison.csv`, `ablation.csv`, `hyperparam_sensitivity.csv`

Send me the contents of the three CSVs and the figures, and I'll write up the report around your actual numbers.